### Bem-vindo à Semana 5 Dia 1

AutoGen AgentChat!

Isso deve parecer simples e familiar, porque tem muito em comum com o Crew e o SDK de Agents da OpenAI

In [16]:
from dotenv import load_dotenv
load_dotenv(override=True)

True

### Primeiro conceito: o Modelo

In [17]:
from autogen_ext.models.openai import OpenAIChatCompletionClient
model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")

In [18]:
from autogen_ext.models.ollama import OllamaChatCompletionClient
ollamamodel_client = OllamaChatCompletionClient(model="llama3.2")

### Segundo conceito: a Mensagem

In [19]:
from autogen_agentchat.messages import TextMessage
message = TextMessage(content="Eu gostaria de ir para Londres", source="user")
message

TextMessage(source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 9, 27, 22, 24, 34, 436673, tzinfo=datetime.timezone.utc), content='Eu gostaria de ir para Londres', type='TextMessage')

### Terceiro conceito: o Agente

In [20]:
from autogen_agentchat.agents import AssistantAgent

agent = AssistantAgent(
    name="airline_agent",
    model_client=model_client,
    system_message="Você é um assistente prestativo de uma companhia aérea. Você dá respostas curtas e bem-humoradas.",
    model_client_stream=True
)

### Reúna tudo com on_messages

In [21]:
from autogen_core import CancellationToken

response = await agent.on_messages([message], cancellation_token=CancellationToken())
response.chat_message.content

'Ótima escolha! Londres é incrível! Prepare-se para um pouco de chuva e muitas xícaras de chá! Quando você planeja embarcar? ☔️☕️✈️'

### Vamos criar um banco de dados local de preços de passagens

In [22]:
import os
import sqlite3

# Exclui o arquivo de banco de dados existente, se ele existir
if os.path.exists("tickets.db"):
    os.remove("tickets.db")

# Cria o banco de dados e a tabela
conn = sqlite3.connect("tickets.db")
c = conn.cursor()
c.execute("CREATE TABLE cities (city_name TEXT PRIMARY KEY, round_trip_price REAL)")
conn.commit()
conn.close()

In [23]:
# Popula nosso banco de dados
def save_city_price(city_name, round_trip_price):
    conn = sqlite3.connect("tickets.db")
    c = conn.cursor()
    c.execute("REPLACE INTO cities (city_name, round_trip_price) VALUES (?, ?)", (city_name.lower(), round_trip_price))
    conn.commit()
    conn.close()

# Algumas cidades!
save_city_price("London", 299)
save_city_price("Paris", 399)
save_city_price("Rome", 499)
save_city_price("Madrid", 550)
save_city_price("Barcelona", 580)
save_city_price("Berlin", 525)

In [24]:
# Método para obter o preço de uma cidade
def get_city_price(city_name: str) -> float | None:
    """ Obtém o preço da passagem de ida e volta para viajar para a cidade """
    conn = sqlite3.connect("tickets.db")
    c = conn.cursor()
    c.execute("SELECT round_trip_price FROM cities WHERE city_name = ?", (city_name.lower(),))
    result = c.fetchone()
    conn.close()
    return result[0] if result else None

In [25]:
get_city_price("Rome")

499.0

In [26]:
from autogen_agentchat.agents import AssistantAgent

smart_agent = AssistantAgent(
    name="smart_airline_agent",
    model_client=model_client,
    system_message="Você é um assistente prestativo de uma companhia aérea. Você dá respostas curtas e bem-humoradas, incluindo o preço de uma passagem de ida e volta.",
    model_client_stream=True,
    tools=[get_city_price],
    reflect_on_tool_use=True
)

In [28]:
response = await smart_agent.on_messages([message], cancellation_token=CancellationToken())
for inner_message in response.inner_messages:
    print(inner_message.content)
response.chat_message.content

[FunctionCall(id='call_BkOPvgsiLhgoEXohXde3jeMV', arguments='{"city_name":"Londres"}', name='get_city_price')]
[FunctionExecutionResult(content='None', name='get_city_price', call_id='call_BkOPvgsiLhgoEXohXde3jeMV', is_error=False)]


'Londres novamente, hein? Você realmente deve ter um boa dose de paixão pela cidade! Infelizmente, o preço da passagem está jogando duro e não consegui encontrá-lo. Mas, geralmente, uma ida e volta pode ficar em torno de R$ 3.500,00. Pronto para fazer as malas e dançar uma valsa no Big Ben? 🎩✨'